# `PP_results.ipynb`

This notebook is for plotting cached `C_\ell^{\phi\phi}` results. It should not be used to recompute the expensive simulation statistics interactively.

## Run Order

1. Run the scenario production jobs with Slurm so the `PLENS` products exist.
2. Run the PP cache-building job so this notebook can load compact `.npz` files.
3. Open this notebook with the `delens-env` Jupyter kernel.
4. Run the cells from top to bottom.

## Required Slurm Job

The notebook assumes the scenario outputs already exist under `$PLENS`.

To build the cached PP plot inputs, run:

```bash
cd /home3/p283342/Delensing/clean-delensing
sbatch compute_pp_plot_data.slurm
```

## What You Must Change On Another Machine Or Cluster

These paths are system-specific and will need to be edited if you run elsewhere:

- `repo_root`: repository location
- `PLENS`: location of the cached Planck lensing products
- `INPUT`: location of the downloaded CMB, noise, and SMICA inputs
- `PARAMS`: location of masks and `dcl_*` files
- `KFIELD`: location of the input lensing maps
- `cache_dir`: location of the `.npz` files written by `compute_pp_plot_data.py`


## Kernel And Path Setup

This cell makes the repository importable from the notebook and sets the environment variables expected by the project code.

If you are running on another machine or cluster, this is the first place you will need to edit paths.


In [1]:
import os
import sys
from pathlib import Path

repo_root = Path('/home3/p283342/Delensing/clean-delensing')
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

os.environ['PLENS'] = '/scratch/hb-CosmoGroup/Delensing/PLENS'
os.environ['INPUT'] = '/scratch/hb-CosmoGroup/Delensing/INPUT'
os.environ['PARAMS'] = '/home3/p283342/Delensing/clean-delensing/input'
os.environ['KFIELD'] = '/scratch/hb-CosmoGroup/Delensing/KFIELD'

print('Python executable:', sys.executable)
print('PLENS:', os.environ.get('PLENS'))
print('INPUT:', os.environ.get('INPUT'))
print('PARAMS:', os.environ.get('PARAMS'))
print('KFIELD:', os.environ.get('KFIELD'))


Python executable: /home3/p283342/delens-env/bin/python
PLENS: /scratch/hb-CosmoGroup/Delensing/PLENS
INPUT: /scratch/hb-CosmoGroup/Delensing/INPUT
PARAMS: /home3/p283342/Delensing/clean-delensing/input
KFIELD: /scratch/hb-CosmoGroup/Delensing/KFIELD


## Imports And Cache Setup

This cell imports the plotting helpers and defines a small cache-loading helper.

Important: the notebook no longer loads all noisy and noiseless caches up front. Each figure cell loads only the files it needs. That means you can work on the noiseless plots before the noisy cache jobs have finished.

If you changed the output directory in the script or the Slurm wrapper, update `cache_dir` here.


In [2]:
import numpy as np
import matplotlib.pyplot as plt

import env_config
import thesis_plot_style as tps

tps.apply_style('single')
cmap = tps.CB_PALETTE

cache_dir = Path('/home3/p283342/Delensing/clean-delensing/THESIS/cache/pp_results')


def load_cache(name: str):
    path = cache_dir / name
    if not path.exists():
        raise FileNotFoundError(
            f"Missing cache file: {path}"
            "Run the corresponding compute_pp_plot_data.py stage first."
        )
    return np.load(path, allow_pickle=True)


## Figure: Noiseless Pipeline Validation

This figure reproduces the basic noiseless `C_\ell^{\phi\phi}` validation check:

- fiducial bandpowers
- the noiseless simulation estimate
- the fractional residual `(sims - fid) / fid`

The expensive simulation average has been cached in `validation_noiseless_agr2.npz`.


In [3]:
validation = load_cache('validation_noiseless_agr2.npz')

ell = validation['ell']
fid = validation['fid']
sims = validation['sims']
nsims = int(validation['nsims'])

with tps.context_two_panel(filename='fig_dual.png', heights=(2/3, 1/3), hspace=0) as (fig, (ax1, ax2)):
    ax1.plot(ell, fid, 'k--', lw=1.6, label=r'Fiducial $C_\ell^{\phi\phi}$')
    ax1.plot(ell, sims, color=tps.CB_PALETTE[0], lw=1.6, label=f'{nsims} noiseless sims')
    ax1.set_ylabel(r'$C_\ell^{\phi\phi}$')
    ax1.legend(loc='upper right')
    ax1.set_title('Lensing Power Spectrum & Residuals')

    frac = (sims - fid) / fid
    ax2.plot(ell, frac, color=tps.CB_PALETTE[1], lw=1.6)
    ax2.axhline(0, color='gray', lw=0.8, ls='--')
    ax2.fill_between([0, 1200], 0.01, -0.01, color='gray', alpha=0.2, zorder=0, label=r'±1\%')
    ax2.set_xlabel(r'$\ell$')
    ax2.set_ylabel(r'$\Delta C_\ell / C_\ell^{\phi\phi}$')
    ax2.set_xlim([0, 1200])


## Figure: Noiseless `C_\ell^{\phi\phi}`

This cell plots the noiseless thesis comparison figure using cached mean bandpowers and their errors for:

- lensed baseline
- delensed with input `\kappa_{LM}`
- internally delensed with MV-QEST
- internally delensed with Pol-QEST


In [4]:
noiseless_fid = load_cache('noiseless_clpp_fiducial_fullA_20.npz')
noiseless_datasets = [
    load_cache('noiseless_clpp_mv_lensed_fullA_20.npz'),
    load_cache('noiseless_clpp_mv_input_kappa_fullA_20.npz'),
    load_cache('noiseless_clpp_mv_internal_qest_fullA_20.npz'),
    load_cache('noiseless_clpp_tt_internal_polqest_fullA_20.npz'),
]

with tps.context_figure(filename='fig_clpp_noiseless.png') as (fig, ax):
    ell = noiseless_fid['ell']
    ax.plot(ell, noiseless_fid['fid'], 'k--', label='fiducial', lw=1.6)

    for ci, d in enumerate(noiseless_datasets):
        ax.errorbar(
            d['ell'],
            d['bp_mean'],
            yerr=d['bp_err'],
            fmt='o-',
            lw=1.6,
            capsize=3,
            markersize=4.5,
            color=tps.CB_PALETTE[ci + 2],
            label=str(d['label']),
        )

    ax.axhline(0, color='gray', lw=0.8, ls='--')
    ax.set(xlabel=r'$\ell$', ylabel=r'$C_\ell^{\phi\phi}$', title='Noiseless $C_\ell^{\phi\phi}$ Estimates')
    ax.set_xlim([0, 400])
    ax.legend(loc='upper right')


FileNotFoundError: Missing cache file: /home3/p283342/Delensing/clean-delensing/THESIS/cache/pp_results/noiseless_clpp_mv_input_kappa_fullA_20.npzRun the corresponding compute_pp_plot_data.py stage first.

## Figure: Noisy `C_\ell^{\phi\phi}`

This is the noisy counterpart of the previous figure. The calibration that was previously computed in the notebook is already folded into the cached noisy datasets.


In [ ]:
noisy_fid = load_cache('noisy_clpp_fiducial_fullA_20.npz')
noisy_datasets = [
    load_cache('noisy_clpp_mv_lensed_fullA_20.npz'),
    load_cache('noisy_clpp_mv_input_kappa_fullA_20.npz'),
    load_cache('noisy_clpp_mv_internal_qest_fullA_20.npz'),
    load_cache('noisy_clpp_tt_internal_polqest_fullA_20.npz'),
]

with tps.context_figure(filename='fig_clpp_noisy.png') as (fig, ax):
    ell = noisy_fid['ell']
    ax.plot(ell, noisy_fid['fid'], 'k--', label='fiducial', lw=1.6)

    for ci, d in enumerate(noisy_datasets):
        ax.errorbar(
            d['ell'],
            d['bp_mean'],
            yerr=d['bp_err'],
            fmt='o-',
            lw=1.6,
            capsize=3,
            markersize=4.5,
            color=tps.CB_PALETTE[ci + 2],
            label=str(d['label']),
        )

    ax.axhline(0, color='gray', lw=0.8, ls='--')
    ax.set(xlabel=r'$\ell$', ylabel=r'$C_\ell^{\phi\phi}$', title='Noisy $C_\ell^{\phi\phi}$ Estimates')
    ax.set_xlim([0, 400])
    ax.legend(loc='upper right')


## Figure: Pol-QE Wiener Filter vs Delensing Efficiency

This cell compares the Pol-QE Wiener filter to the delensing efficiency for the noiseless and noisy cases.

The expensive `\phi`-`\phi` bandpower and efficiency calculations have been cached, so the notebook only reads compact `.npz` files here.


In [ ]:
wf_eff_noiseless = load_cache('noiseless_wf_vs_eff_fullA_10.npz')
wf_eff_noisy = load_cache('noisy_wf_vs_eff_fullA_10.npz')

with tps.context_figure(filename='fig_wf_vs_eff_combined.png') as (fig, ax):
    ax.axvspan(25, 35, color='gray', alpha=0.2, zorder=0)

    for i, (label, data) in enumerate([
        ('Noiseless', wf_eff_noiseless),
        ('Noisy', wf_eff_noisy),
    ]):
        ax.plot(
            data['ell_w'],
            data['w'],
            color=cmap[2 + 2 * i],
            lw=1.6,
            label=f'Wiener $W_\ell$ ({label})',
        )
        ax.errorbar(
            data['ell'],
            data['eff_mean'],
            yerr=data['eff_err'],
            fmt='o',
            ls='--',
            lw=1.2,
            capsize=3,
            color=cmap[3 + 2 * i],
            label=f'Eff. $1 - C_\ell^{{\rm del}}/C_\ell^{{\rm len}}$ ({label})',
        )

    ax.hlines(0, 0, 300, 'grey', '--', alpha=0.3)
    ax.set_xlim(0, 300)
    ax.set_xlabel(r'$\ell$')
    ax.set_ylabel('Dimensionless')
    ax.set_ylim([-.1, 0.6])
    ax.set_title('Pol-QE Wiener Filter vs Delensing Efficiency\n(Noiseless & Noisy)')
    ax.legend(loc='upper right', frameon=False)
